# Data Loading Demo

This notebook demonstrates the data loading and validation functionality of the BoneStrengthML pipeline.

In [1]:
from bonestrength_ml.config import load_config
from bonestrength_ml.data_loading import (
    load_raw_data,
    validate_data,
    load_and_validate_data,
    build_schema_from_config,
    get_input_columns,
    get_output_columns,
    DataValidationError,
)

## 1. Load Configuration

The configuration file defines dataset metadata, field specifications, and validation rules.

In [2]:
config = load_config()

print(f"Dataset: {config.dataset.name}")
print(f"Version: {config.dataset.version}")
print(f"Format: {config.dataset.file_format}")
print(f"Delimiter: '{config.dataset.format_metadata.delimiter}'")

Dataset: BoneStrengthML-dataset
Version: v1
Format: CSV
Delimiter: ';'


## 2. Inspect Field Specifications

The config defines input/output fields with types, ranges, and constraints.

In [3]:
input_cols = get_input_columns(config)
output_cols = get_output_columns(config)

print(f"Input columns: {len(input_cols)}")
print(f"Output columns: {len(output_cols)}")
print(f"\nOutputs: {output_cols}")

Input columns: 95
Output columns: 2

Outputs: ['maxStrain_11', 'maxStrain_33']


In [4]:
# Show first few input field specs
print("Sample input field specifications:")
for field in config.dataset.inputs[:3]:
    print(f"  {field.name}: {field.data_type}, range={field.range}, nullable={field.missing_values_allowed}")

print("\nOutput field specifications:")
for field in config.dataset.outputs:
    print(f"  {field.name}: {field.data_type}, range={field.range}, nullable={field.missing_values_allowed}")

Sample input field specifications:
  PC_1: float, range=(-1877424.1, 1877424.1), nullable=False
  PC_2: float, range=(-922418.6, 922418.6), nullable=False
  PC_3: float, range=(-685300.0, 685300.0), nullable=False

Output field specifications:
  maxStrain_11: float, range=(0.0, 0.008), nullable=False
  maxStrain_33: float, range=(0.0, 0.03), nullable=False


## 3. Load Raw Data

Load the dataset using format metadata from configuration.

In [5]:
df = load_raw_data(config)

print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

Shape: (45332, 97)
Memory usage: 35.2 MB


,PC_1,PC_2,PC_3,PC_4,PC_5,PC_6,PC_7,PC_8,PC_9,PC_10,...,PC_88,PC_89,PC_90,PC_91,PC_92,PC_93,PosAnt,MedLat,maxStrain_11,maxStrain_33
0,-418340.211279,-122588.930769,-71226.839864,-181297.595216,33883.792590,196747.208561,-104744.439674,-20441.424380,-67417.357696,9622.702832,...,19730.274527,38571.471859,13897.279219,11606.866959,-13549.019341,2273.316613,2,-19,0.001459,0.003854
1,-596726.737283,-52801.874009,-63976.004793,222195.033929,104194.380612,249311.861782,-108596.466874,-100452.105056,123610.780181,-202626.247381,...,-20644.091040,7459.355675,-7663.700411,3660.449382,8568.906644,-4155.710429,-3,-1,0.001937,0.005644
2,-364061.661850,20905.288290,-284898.238405,-364501.237318,233438.818485,-333502.146591,-121962.067164,-157065.279018,20692.875507,133953.450360,...,23165.064069,27790.740528,4339.492432,-15981.088318,-740.971475,9023.209300,11,-29,0.001244,0.003528
3,443174.461425,164855.829920,-277166.479254,-364560.126579,82574.304353,-121398.087678,53530.136182,12322.898345,-122084.548780,-2378.096637,...,-15340.529120,-15283.798661,-6086.533750,-4200.097149,-8377.870394,-3877.745417,15,-6,0.001390,0.002628
4,-235475.229552,-233153.519066,-15225.657530,-79967.426173,-21632.961609,-251580.977289,132545.095591,-16856.476265,115324.821834,165197.955828,...,17.513339,6377.311513,11923.157113,-12496.046749,-2422.201657,-1079.026499,27,-5,0.002281,0.003415


In [6]:
df.describe()

,PC_1,PC_2,PC_3,PC_4,PC_5,PC_6,PC_7,PC_8,PC_9,PC_10,...,PC_88,PC_89,PC_90,PC_91,PC_92,PC_93,PosAnt,MedLat,maxStrain_11,maxStrain_33
count,4.533200e+04,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,...,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000,45332.000000
mean,6.823347e+03,8338.786111,-3927.154975,-13055.845092,-14852.712685,-14240.335274,-7830.671271,1239.483679,-1489.666972,-6050.149843,...,-697.534826,-134.683729,217.061579,61.355809,2478.722320,-328.233157,0.029582,-14.992566,0.001961,0.004423
std,5.868637e+05,271361.027910,196849.032243,182235.236390,180143.034887,156740.267225,124120.406243,124310.257916,116357.448833,110479.871763,...,31820.594282,34065.527110,21092.978065,20522.973277,15006.814570,6861.083973,17.613935,8.929477,0.000667,0.001588
min,-1.122333e+06,-725174.271225,-506930.198587,-403807.131698,-421654.095459,-343448.613514,-333246.883748,-325758.859537,-252384.568018,-271060.341049,...,-106828.973501,-120936.237964,-73945.144921,-61450.379153,-39129.431212,-30137.757089,-30.000000,-30.000000,0.000685,0.001657
25%,-4.857621e+05,-122588.930769,-161091.798733,-141101.651979,-156593.016158,-122182.214151,-94610.246527,-92173.264274,-93923.947974,-82493.898658,...,-17666.120916,-12275.473981,-8519.224644,-10689.297126,-9011.945170,-4168.854480,-15.000000,-23.000000,0.001496,0.003343
50%,-2.547721e+04,18506.167011,10958.784783,-9614.269203,-18619.273167,-11247.811981,-11884.569143,-4580.532395,-5332.814668,7526.023075,...,3327.668966,-706.552950,1517.178031,1631.173228,245.949618,-103.771458,0.000000,-15.000000,0.001823,0.004091
75%,4.480396e+05,178583.773395,155122.882937,90741.330392,110720.658853,105184.105678,77207.581795,82039.083645,97869.619440,70914.221663,...,19927.013027,19281.941887,11209.951447,10841.929542,10610.695882,3426.210193,15.000000,-7.000000,0.002264,0.005100
max,1.275087e+06,608105.008204,487225.286727,415847.894979,430568.947082,312090.868086,311296.433778,302184.612548,236109.424440,281187.125192,...,92265.796404,93480.660525,72503.699964,53185.974154,46214.539613,15443.376350,30.000000,0.000000,0.007365,0.019288


## 4. Build Validation Schema

Pandera schema is dynamically generated from the configuration.

In [7]:
schema = build_schema_from_config(config.dataset)

print(f"Schema columns: {len(schema.columns)}")
print(f"Strict mode: {schema.strict}")
print(f"\nSample column schema (PC_1):")
print(schema.columns["PC_1"])

Schema columns: 97
Strict mode: True

Sample column schema (PC_1):
<Schema Column(name=PC_1, type=DataType(float64))>


## 5. Validate Data

Run validation to check data against configuration constraints.

In [8]:
# Check actual vs configured ranges for outputs
print("Range comparison (config vs actual):")
print("=" * 60)

for field in config.dataset.outputs:
    actual_min = df[field.name].min()
    actual_max = df[field.name].max()
    cfg_min, cfg_max = field.range

    in_range = (df[field.name] >= cfg_min) & (df[field.name] <= cfg_max)
    pct_valid = in_range.mean() * 100

    print(f"\n{field.name}:")
    print(f"  Config range: [{cfg_min}, {cfg_max}]")
    print(f"  Actual range: [{actual_min:.6f}, {actual_max:.6f}]")
    print(f"  Valid rows: {pct_valid:.1f}%")

Range comparison (config vs actual):

maxStrain_11:
  Config range: [0.0, 0.008]
  Actual range: [0.000685, 0.007365]
  Valid rows: 100.0%

maxStrain_33:
  Config range: [0.0, 0.03]
  Actual range: [0.001657, 0.019288]
  Valid rows: 100.0%


In [9]:
# Attempt validation
try:
    validated_df = validate_data(df, config)
    print("Validation passed!")
    print(f"Validated shape: {validated_df.shape}")
except DataValidationError as e:
    print("Validation failed (expected if config ranges don't match data):")
    print(f"  Failure cases: {len(e.failure_cases)}")
    print(f"  Columns with issues: {e.failure_cases['column'].unique().tolist()}")

Validation passed!
Validated shape: (45332, 97)


## 6. One-Step Load and Validate

For production use, `load_and_validate_data()` combines both steps.

In [10]:
# This will raise DataValidationError if validation fails
# Uncomment to test after fixing config ranges

# df = load_and_validate_data()
# print(f"Loaded and validated: {df.shape}")

## 7. Working with Inputs and Outputs

In [11]:
# Separate features (X) and targets (y)
X = df[get_input_columns(config)]
y = df[get_output_columns(config)]

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")
print(f"\nTarget statistics:")
y.describe()

Features shape: (45332, 95)
Targets shape: (45332, 2)

Target statistics:


,maxStrain_11,maxStrain_33
count,45332.000000,45332.000000
mean,0.001961,0.004423
std,0.000667,0.001588
min,0.000685,0.001657
25%,0.001496,0.003343
50%,0.001823,0.004091
75%,0.002264,0.005100
max,0.007365,0.019288
